# Verifying Symbolic-Rotation Circuits with `PhasedOutcomeCompleteSimulation`

This notebook shows how to apply a **symbolic rotation** $e^{i\alpha P}$ to a
`PhasedOutcomeCompleteSimulation` and use it to verify equivalence of non-stabilizer circuits, the
motivating application of [arXiv:2603.24717](https://arxiv.org/abs/2603.24717). (See the companion
notebook *Tracking the Exact Global Phase* for an introduction to the phased simulator itself.)

## Modeling a symbolic rotation

An arbitrary-angle rotation $e^{i\alpha P} = \cos(\alpha)\,I + i\sin(\alpha)\,P$ is **not** a
stabilizer operation, so it cannot be applied directly. (Note that `apply_pauli_exp` is only the
*fixed* Clifford rotation $e^{i\pi/4\,P}$, not a free-angle rotation.)

Following §4.1 of the paper, we instead apply the Pauli $P$ **conditioned on a fresh random bit**
$a$. The outcome-complete machinery then tracks both branches at once:

- $a = 0$: the identity branch, carrying amplitude weight $\cos\alpha$,
- $a = 1$: the $P$ branch, carrying amplitude weight $i\sin\alpha$,

so that $e^{i\alpha P}|\psi\rangle = \cos(\alpha)\,|\text{branch }0\rangle + i\sin(\alpha)\,|\text{branch }1\rangle$
for **every** $\alpha$. Because the phased simulator keeps the *exact* global phase of each branch,
two circuits agree for all $\alpha$ iff their branch states match exactly — phase included.

Nothing here forms a $2^n$ state vector: phases are exact integer powers of $\zeta_8 = e^{i\pi/4}$ and
every step is polynomial in the number of qubits.

## Setup

In [1]:
from paulimer import (
    PhasedOutcomeCompleteSimulation,
    OutcomeCompleteSimulation,
    SparsePauli,
    UnitaryOpcode,
)

ZETA8_LABEL = {0: "1", 1: "ζ₈", 2: "i", 3: "ζ₈³", 4: "-1", 5: "ζ₈⁵", 6: "-i", 7: "ζ₈⁷"}

## A single symbolic rotation, two branches

We build $C_1\, e^{i\alpha Z_0}\, C_2\,|0\rangle$ with $C_2 = H_0$ and $C_1 = H_0$, modeling the
rotation by a `Z_0` conditioned on a fresh random bit.

In [2]:
sim = PhasedOutcomeCompleteSimulation(1)
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_2
a = sim.allocate_random_bit()                               # the rotation's random bit
sim.apply_conditional_pauli(SparsePauli("Z_0"), [a])        # e^{iα Z_0}  ->  Z_0^a
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_1

print("random outcomes:", sim.random_outcome_count)
for value in (0, 1):
    e = sim.output_phase_exponent([bool(value)])
    print(f"  branch a={value}:  zeta8^{e} = {ZETA8_LABEL[e]}"
          f"   (amplitude weight {'cos α' if value == 0 else 'i·sin α'})")

random outcomes: 1
  branch a=0:  zeta8^0 = 1   (amplitude weight cos α)
  branch a=1:  zeta8^0 = 1   (amplitude weight i·sin α)


## Verifying two implementations are equivalent

Since $H\,e^{i\alpha Z}\,H = e^{i\alpha X}$, the rotation above should be exactly equivalent to
applying $e^{i\alpha X_0}$ with no surrounding Cliffords. We capture the simulator's full exact
**phase signature** — the encoder (with its symplectic action), the sign matrix $A$, the quadratic
phase matrix $B$, the linear $i$/$-1$ phase vectors $p, s$, and the per-branch $\zeta_8$ exponents —
and check that the two implementations agree.

In [3]:
def phased_signature(build_gadget):
    sim = PhasedOutcomeCompleteSimulation(1)
    a = sim.allocate_random_bit()
    build_gadget(sim, a)
    return {
        "clifford": str(sim.clifford),
        "A": str(sim.sign_matrix),
        "B": str(sim.quadratic_phase_matrix),
        "p": str(sim.linear_i_phase),
        "s": str(sim.linear_sign_phase),
        "branch_phases": tuple(sim.output_phase_exponent([bool(v)]) for v in (0, 1)),
    }

def reference(sim, a):           # H · e^{iα Z0} · H
    sim.apply_unitary(UnitaryOpcode.Hadamard, [0])
    sim.apply_conditional_pauli(SparsePauli("Z_0"), [a])
    sim.apply_unitary(UnitaryOpcode.Hadamard, [0])

def equivalent(sim, a):          # e^{iα X0}
    sim.apply_conditional_pauli(SparsePauli("X_0"), [a])

assert phased_signature(reference) == phased_signature(equivalent)
print("✓ reference (H e^{iα Z} H) and equivalent (e^{iα X}) have identical phase signatures")
print("  branch phases:", phased_signature(reference)["branch_phases"])

✓ reference (H e^{iα Z} H) and equivalent (e^{iα X}) have identical phase signatures
  branch phases: (0, 0)


## Catching a phase bug that ordinary simulation misses

Now consider a buggy implementation that uses $e^{i\alpha Y_0}$ instead of $e^{i\alpha X_0}$. These
two rotations have the **same symplectic (phaseless) action**, so an ordinary stabilizer simulation
cannot tell them apart — yet they are physically different operations, differing by a relative phase
between the branches. The phased simulator detects exactly this.

In [4]:
def buggy(sim, a):               # e^{iα Y0}  -- same symplectic action, different phase
    sim.apply_conditional_pauli(SparsePauli("Y_0"), [a])

ref_sig = phased_signature(reference)
bug_sig = phased_signature(buggy)

assert ref_sig != bug_sig, "phased simulator should distinguish these"
print("Phased simulator:")
print("  reference branch phases:", ref_sig["branch_phases"])
print("  buggy     branch phases:", bug_sig["branch_phases"], "  <- relative i-phase on a=1")
assert ref_sig["clifford"] == bug_sig["clifford"]
print("  (symplectic encoders are identical:", ref_sig["clifford"], ")")

Phased simulator:
  reference branch phases: (0, 0)
  buggy     branch phases: (0, 2)   <- relative i-phase on a=1
  (symplectic encoders are identical: Z₀→Z₀, X₀→X₀ )


In [5]:
def plain_signature(build_gadget):
    sim = OutcomeCompleteSimulation(1)
    a = sim.allocate_random_bit()
    build_gadget(sim, a)
    return {
        "clifford": str(sim.clifford),
        "A": str(sim.sign_matrix),
        "M": str(sim.outcome_matrix),
        "v0": str(sim.outcome_shift),
    }

assert plain_signature(reference) == plain_signature(buggy)
print("Ordinary OutcomeCompleteSimulation: reference and buggy are INDISTINGUISHABLE")
print("  (it discards the global phase, so e^{iα X} and e^{iα Y} look identical)")

Ordinary OutcomeCompleteSimulation: reference and buggy are INDISTINGUISHABLE
  (it discards the global phase, so e^{iα X} and e^{iα Y} look identical)


## Summary

- A symbolic rotation $e^{i\alpha P}$ is applied by conditioning the Pauli $P$ on a fresh
  `allocate_random_bit()` and calling `apply_conditional_pauli(P, [a])`.
- The phased simulator tracks both branches with their **exact** global phase, which is precisely
  what is needed to verify equivalence of non-stabilizer circuits across all angles $\alpha$.
- That exact phase is information ordinary stabilizer simulation throws away: here it is the only
  thing distinguishing $e^{i\alpha X}$ from $e^{i\alpha Y}$.

The comparison above checks the exact phase data the simulator exposes — in particular the relative
phases between measurement branches. A fully canonical equality test that also pins down the
encoder's *absolute* global phase uses the auxiliary-qubit separation of §4.5, a planned follow-up.
As always, no state vector is ever materialized; all phases are exact integer $\zeta_8$ exponents.